# 02 — LLM-as-Judge Evaluator

**The value.** Some qualities are easy to describe but painful to code — "does the response stay in character", "is the tone professional but warm", "does the answer satisfy all the constraints in the user's prompt". LLM-as-Judge lets you express the metric in natural language and have another Bedrock model grade each candidate. You skip writing, deploying, and maintaining metric code.

**Why this method.** Pick a judge when the dimension you care about is subjective, perceptual, or compositional — anything where you'd find it faster to write a paragraph of rubric than a hundred lines of scoring logic. The trade-off: judge calls cost Bedrock tokens, and a weak judge produces noisy scores. Use a strong judge (Claude Sonnet/Opus class) for high-stakes optimizations.

**How it works.** You author a `customLLMJPrompt` rubric with `{prompt}`, `{prediction}`, and `{gold}` placeholders, and pick a `customLLMJModelId`. For every candidate × sample, APO formats the rubric with those three pieces and asks the judge for a numeric score. The score is aggregated and drives the same optimization loop as Lambda mode — only the scoring backend changes.

**You will learn**
- How to author a rubric that doesn't trip the harness's `str.format` parser.
- How `customLLMJConfig.{customLLMJPrompt, customLLMJModelId}` slots into the record.
- How vision-capable judges score image samples.

**Two Scenarios**
1. **Text** — IFBench (instruction-following constraints).
2. **Image** — Defactify (real vs AI-generated images with a composite label+emoji rubric).

> **Tip:** set `APO_USE_REFERENCE=1` to replay both chapters from bundled results.

In [ ]:
import utils
from IPython.display import display, Markdown
utils.render_mode_overview("llmj")

In [ ]:
import os
from pathlib import Path
from IPython.display import display, Markdown

import utils

env = utils.load_env()
bedrock, s3, lambda_client = utils.make_clients(env)

# Default target model matches the bundle reference run. Edit and re-execute to try other models.
TARGET_MODEL_ID = "moonshotai.kimi-k2.5"
JUDGE_MODEL_ID = "us.anthropic.claude-opus-4-6-v1"

# Replay-mode toggle. Set `APO_USE_REFERENCE=1` before launching Jupyter to skip live AWS calls.
USING_REPLAY = os.environ.get("APO_USE_REFERENCE") == "1"
display(Markdown(f"**Live mode:** `{not USING_REPLAY}` &nbsp;|&nbsp; **Bucket:** `{env['BUCKET']}` &nbsp;|&nbsp; **Region:** `{env['REGION']}`"))

---
## Chapter A — Text (IFBench)

The IFBench dataset asks the model to follow strict formatting constraints (exact word counts, bullet structure, keyword frequency). The judge scores the fraction of constraints actually satisfied.

> **Brace-escaping gotcha.** The judge harness applies `str.format(prompt=..., prediction=..., gold=...)` to your rubric. Literal `{` / `}` inside the rubric body must be escaped as `{{` / `}}` or the format call raises `KeyError` and every sample silently scores 0.0.

In [ ]:
IFBENCH_RUBRIC = (
    "You are evaluating whether the model's response satisfies the user's instruction-following constraints.\n\nScore the response on ONE dimension:\n  - Constraint Adherence (weight: 1.0): the fraction of explicit formatting and content constraints in the user's instruction that the response actually satisfies. Examples: exact keyword frequency, structural rules, content limits. The reference answer encodes the expected constraints in JSON form.\n\nScore 1 if the response satisfies all constraints, 0 if none, fractional in between."
)
print(IFBENCH_RUBRIC[:400] + " …")

In [ ]:
preview = utils.build_llmj_record(
    "ifbench",
    rubric=IFBENCH_RUBRIC,
    judge_model_id=JUDGE_MODEL_ID,
    bucket=env["BUCKET"],
)
display(utils.render_dimensions(preview))
display(Markdown("**First sample shape:**"))
display(utils.render_sample_shape(preview))

> **Heads up — long-running cell.** The call below submits a real APO job that can take up to ~20 minutes per chapter. The cell will keep its `status` / `elapsed` display refreshing until the job reaches a terminal state (`Completed`, `Failed`, or `Stopped`). Safe to leave running while you work on something else.

In [ ]:
record = utils.build_llmj_record("ifbench", rubric=IFBENCH_RUBRIC, judge_model_id=JUDGE_MODEL_ID, bucket=env["BUCKET"])
display(utils.render_record_shape(record))

handle = display(Markdown("_waiting for status…_"), display_id=True)
def on_status(status, elapsed_s):
    handle.update(Markdown(f"**status:** `{status}` &nbsp;|&nbsp; **elapsed:** `{elapsed_s:.0f}s`"))

ifbench_results = utils.run_job(
    record,
    example="ifbench",
    model_id=TARGET_MODEL_ID,
    env=env,
    on_status=on_status,
)
print(f"results file: {ifbench_results}")

In [ ]:
parsed = utils.parse_results(ifbench_results)
display(utils.render_results_table(parsed))
for row in parsed:
    display(utils.render_prompt_diff(row))

**A note on polling.** `utils.run_job(...)` orchestrates upload → submit → poll → download. We pass it an `on_status` callback that overwrites a single Markdown line in place — no scrolling log spam over the 20–50 minute job run.

If `APO_USE_REFERENCE=1` is set, `run_job` returns the bundled `reference_results.jsonl` without calling AWS.

---
## Chapter B — Image (Defactify)

The model is shown an image and a caption, asked to classify it Real vs AI-Generated. The rubric weights two criteria equally: label correctness (deterministic) plus an emoji visual-match (perceptual). Only an LLM judge can score the second half.

You'll need a vision-capable **target** model and a vision-capable **judge** model — the judge sees the same image as the target.

In [ ]:
# Swap target to a vision-capable model.
TARGET_MODEL_ID = "us.anthropic.claude-sonnet-4-6"

# Sync images to S3 (idempotent).
if not USING_REPLAY:
    utils.sync_assets("defactify", env, s3=s3)

In [ ]:
DEFACTIFY_RUBRIC = (
    "You are evaluating a vision model's classification response. The model was given an image and a caption, and asked to determine whether the image is Real or AI-Generated. The reference label is provided.\n\nScore on TWO equally-weighted criteria (50% each):\n  1. Label correctness: Does the response's verdict match the reference?\n  2. Emoji visual match: Does the response include at least one emoji that visually resembles the dominant subject? (cat photo + 🐈 = match; chart + 📊 = match.)\n\nscore = 0.5 * label_correctness + 0.5 * emoji_match, each sub-score in {{0, 1}}.\n\nOutput exactly two lines:\n  Line 1: `Score: <float in [0, 1]>`\n  Line 2: a one-sentence rationale that names the verdict, the gold label, and the emoji (or notes its absence)."
)

preview = utils.build_llmj_record(
    "defactify",
    rubric=DEFACTIFY_RUBRIC,
    judge_model_id=JUDGE_MODEL_ID,
    bucket=env["BUCKET"],
)
display(utils.render_dimensions(preview))
display(Markdown("**First sample shape (`inputVariablesMultimodal` carries the image):**"))
display(utils.render_sample_shape(preview))

> **Heads up — long-running cell.** The call below submits a real APO job that can take up to ~20 minutes per chapter. The cell will keep its `status` / `elapsed` display refreshing until the job reaches a terminal state (`Completed`, `Failed`, or `Stopped`). Safe to leave running while you work on something else.

In [ ]:
record = utils.build_llmj_record("defactify", rubric=DEFACTIFY_RUBRIC, judge_model_id=JUDGE_MODEL_ID, bucket=env["BUCKET"])
display(utils.render_record_shape(record))

handle = display(Markdown("_waiting for status…_"), display_id=True)
def on_status(status, elapsed_s):
    handle.update(Markdown(f"**status:** `{status}` &nbsp;|&nbsp; **elapsed:** `{elapsed_s:.0f}s`"))

defactify_results = utils.run_job(
    record,
    example="defactify",
    model_id=TARGET_MODEL_ID,
    env=env,
    on_status=on_status,
)
print(f"results file: {defactify_results}")

In [ ]:
parsed = utils.parse_results(defactify_results)
display(utils.render_results_table(parsed))
for row in parsed:
    display(utils.render_prompt_diff(row))

---
## Recap

You authored two rubrics (one text-only, one multimodal), submitted via boto3 with `customLLMJConfig`, and observed the judge-driven score lift.

**Next:** `03_steering_criteria.ipynb` — no Lambda, no judge model. You just give the optimizer a list of plain-English rules.